In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 1. Local PDF Q&A Tutor

## What We're Building
A **Retrieval-Augmented Generation (RAG)** pipeline that lets you chat with any PDF
document using 100% local models via Ollama — no cloud APIs needed.

**You will learn:**
- How to load and chunk PDF documents
- How vector embeddings work for semantic search
- How to build a full RAG pipeline with LangChain + ChromaDB
- How to use Ollama for both embeddings and LLM inference

## Architecture
```
PDF → Extract text → Split into chunks → Embed chunks → Store in ChromaDB
                                                              ↓
User Question → Embed question → Retrieve similar chunks → LLM generates answer
```

**Stack:** Ollama, LangChain, ChromaDB, PyPDF

## Step 1 — Install Dependencies

In [ ]:
# Install required packages (uncomment to run)
# !pip install -q langchain langchain-ollama langchain-community chromadb pypdf

import warnings
warnings.filterwarnings("ignore")
print("Ready to go!")

## Step 2 — Imports and Configuration

We'll set up all our imports and configure the Ollama connection.
The LLM runs on `localhost:11434` — make sure Ollama is running.

In [ ]:
import os
from pathlib import Path

# LangChain core
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Configuration
OLLAMA_MODEL = "qwen3:8b"       # Change to any model you have pulled
EMBED_MODEL = "nomic-embed-text"  # Embedding model
OLLAMA_BASE = "http://localhost:11434"

print("All imports loaded successfully!")

## Step 3 — Initialize Local LLM and Embeddings

**What are embeddings?**
Embeddings convert text into numerical vectors (lists of numbers). Similar
meanings produce vectors that are close together in space. This enables
*semantic search* — finding text by meaning, not just keyword matching.

**Why local?**
Running models locally means your data never leaves your machine. No API keys,
no costs, no privacy concerns.

In [ ]:
# Initialize the LLM — runs entirely on your local machine
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0.2,     # Lower = more focused, factual answers
    num_predict=1024,    # Max tokens to generate
)

# Initialize the embedding model
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
)

# Quick test — embed a sentence and show the vector shape
test_vector = embeddings.embed_query("What is machine learning?")
print(f"Embedding model: {EMBED_MODEL}")
print(f"Vector dimensions: {len(test_vector)}")
print(f"First 5 values: {[round(v, 4) for v in test_vector[:5]]}")

## Step 4 — Create a Sample PDF (for demo purposes)

In a real project, you'd point this at your own PDFs. Here we'll create a
small sample PDF so the notebook is self-contained and runnable.

In [ ]:
# Create a sample PDF for demonstration
SAMPLE_DIR = Path("sample_data")
SAMPLE_DIR.mkdir(exist_ok=True)

sample_text = """
Machine Learning Fundamentals

Chapter 1: Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn
and improve from experience without being explicitly programmed. It focuses on the
development of computer programs that can access data and use it to learn for themselves.

There are three main types of machine learning:
1. Supervised Learning: The algorithm learns from labeled training data.
2. Unsupervised Learning: The algorithm finds patterns in unlabeled data.
3. Reinforcement Learning: The algorithm learns by interacting with an environment.

Chapter 2: Supervised Learning

In supervised learning, we have a dataset with input-output pairs. The goal is to learn
a mapping function from inputs to outputs. Common algorithms include:
- Linear Regression: Predicts continuous values
- Logistic Regression: Predicts categorical outcomes
- Decision Trees: Tree-based classification and regression
- Random Forests: Ensemble of decision trees
- Support Vector Machines: Finds optimal hyperplanes for classification

Chapter 3: Model Evaluation

Key metrics for evaluating ML models:
- Accuracy: Proportion of correct predictions
- Precision: True positives / (True positives + False positives)
- Recall: True positives / (True positives + False negatives)
- F1 Score: Harmonic mean of precision and recall
- AUC-ROC: Area under the Receiver Operating Characteristic curve

Cross-validation is essential for reliable model evaluation. K-fold cross-validation
splits data into K subsets, training on K-1 and testing on the remaining fold.
"""

# Write as a text file (simulates PDF content for demo)
sample_path = SAMPLE_DIR / "ml_fundamentals.txt"
sample_path.write_text(sample_text)
print(f"Sample document created at: {sample_path}")
print(f"Document length: {len(sample_text)} characters")

## Step 5 — Load and Chunk the Document

**Why chunk?**
- LLMs have limited context windows
- Smaller chunks = more precise retrieval
- We can cite exactly which chunk answered the question

**Chunking strategy:**
We use `RecursiveCharacterTextSplitter` which tries to split on natural boundaries
(paragraphs → sentences → words) to keep semantic coherence.

In [ ]:
from langchain_community.document_loaders import TextLoader

# Load the document
# For real PDFs, use: loader = PyPDFLoader("path/to/file.pdf")
loader = TextLoader(str(SAMPLE_DIR / "ml_fundamentals.txt"), encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"Total characters: {sum(len(d.page_content) for d in documents)}")

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Characters per chunk
    chunk_overlap=50,      # Overlap between chunks (prevents cutting mid-sentence)
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Priority order for splitting
)

chunks = text_splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

# Show first chunk
print(f"\n--- Chunk 1 Preview ---")
print(chunks[0].page_content[:200])

## Step 6 — Build the Vector Store

ChromaDB stores our chunks as vectors for fast similarity search.
When a user asks a question, we embed the question and find the most
similar chunks — this is the "retrieval" in RAG.

In [ ]:
# Create ChromaDB vector store from our chunks
# This embeds every chunk using our local Ollama embedding model
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pdf_qa_tutor",
    persist_directory=str(SAMPLE_DIR / "chroma_db"),
)

# Create a retriever (fetches top-k most relevant chunks)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # Return top 3 most relevant chunks
)

# Test retrieval
test_query = "What are the types of machine learning?"
relevant_docs = retriever.invoke(test_query)
print(f"Query: {test_query}")
print(f"Retrieved {len(relevant_docs)} chunks:\n")
for i, doc in enumerate(relevant_docs, 1):
    print(f"--- Chunk {i} (score: relevance-ranked) ---")
    print(doc.page_content[:150])
    print()

## Step 7 — Build the Q&A Chain

Now we combine retrieval with generation. The chain will:
1. Take a user question
2. Retrieve relevant chunks from the vector store
3. Feed the question + context to the LLM
4. Return a grounded answer

In [ ]:
# Define the prompt template — this tells the LLM how to use retrieved context
qa_prompt = PromptTemplate(
    template="""You are a helpful tutor. Use the following context from the document
to answer the question. If the context doesn't contain the answer, say so honestly.

Context:
{context}

Question: {question}

Provide a clear, educational answer based on the context above.""",
    input_variables=["context", "question"],
)

# Build the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # "stuff" = put all retrieved docs into prompt
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": qa_prompt},
)

print("Q&A chain ready! Let's ask some questions.")

## Step 8 — Ask Questions!

In [ ]:
# Ask questions about the document
questions = [
    "What is machine learning?",
    "What are the three types of machine learning?",
    "How do you evaluate an ML model?",
    "What is cross-validation?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    result = qa_chain.invoke({"query": q})
    print(f"A: {result['result']}")
    print(f"\n  Sources used: {len(result['source_documents'])} chunks")

## Step 9 — Interactive Q&A Loop

In [ ]:
# Interactive Q&A — ask your own questions
def ask(question: str) -> str:
    """Ask a question about the loaded document."""
    result = qa_chain.invoke({"query": question})
    answer = result["result"]
    sources = result["source_documents"]

    print(f"\nAnswer: {answer}")
    print(f"\n--- Sources ({len(sources)} chunks used) ---")
    for i, doc in enumerate(sources, 1):
        print(f"  [{i}] {doc.page_content[:100]}...")
    return answer

# Try it:
ask("What algorithms are used in supervised learning?")

## Summary & Next Steps

**What you learned:**
- ✅ Loading and chunking documents for RAG
- ✅ Creating vector embeddings with a local model
- ✅ Building a ChromaDB vector store
- ✅ Creating a RetrievalQA chain with LangChain
- ✅ Running everything locally with Ollama

**Next steps:**
- Try with real PDFs using `PyPDFLoader`
- Experiment with different `chunk_size` and `chunk_overlap` values
- Try different Ollama models (llama3, mistral, phi3)
- Add metadata filtering for multi-document collections
- Move to Project 2 for LlamaIndex-based approach

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
